In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, ElasticNet
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy import stats

class AdvancedRealEstatePreprocessor:
    def __init__(self, district_prices_path):
        self.district_prices = pd.read_csv(district_prices_path, sep=';')
        self.preprocessor = None
        self.feature_names = None

    def create_advanced_features(self, df):
        """Fejlett feature engineering - KIADÓ SPECIFIKUS"""
        df = df.copy()
        
        # ===== ALAPVETŐ FEATURE-ÖK =====
        
        # 1. Szobánkénti négyzetméter (ha van rooms és area_m2 adat)
        if 'area_m2' in df.columns and 'rooms' in df.columns:
            df['m2_per_room'] = df['area_m2'] / df['rooms'].replace(0, np.nan)
        else:
            df['m2_per_room'] = np.nan
        
        # 2. Erkély/Terasz boolean-ok
        def parse_area(area_str):
            if pd.isna(area_str):
                return 0.0
            try:
                clean_str = re.sub(r'[^\\d,\\.]', '', str(area_str)).replace(',', '.')
                return float(clean_str)
            except (ValueError, TypeError):
                return 0.0

        if 'Erkély' in df.columns:
            df['balcony_area'] = df['Erkély'].apply(parse_area)
            df['has_balcony'] = (df['balcony_area'] > 0).astype(int)
        else:
            df['balcony_area'] = 0.0
            df['has_balcony'] = 0

        if 'Terasz' in df.columns:
            df['terrace_area'] = df['Terasz'].apply(parse_area)
            df['has_terrace'] = (df['terrace_area'] > 0).astype(int)
        else:
            df['terrace_area'] = 0.0
            df['has_terrace'] = 0

        df['outdoor_space'] = df['balcony_area'] + df['terrace_area']
        
        # 3. Építés éve alapú kategóriák
        if 'Építés éve' in df.columns:
            df['building_age'] = 2024 - df['Építés éve'].fillna(2000)
            df['era'] = pd.cut(df['Építés éve'].fillna(2000), 
                                bins=[0, 1945, 1970, 1990, 2010, 2024], 
                                labels=['Előháború', 'Szocializmus', 'Rendszerváltás', 'Modern', 'Új'])
        else:
            df['building_age'] = np.nan
            df['era'] = 'Ismeretlen'
        
        # 4. Emelet kategóriák
        if 'Szintek száma' in df.columns:
            df['is_ground_floor'] = (df['floor_numeric'] == 0).astype(int)
            df['is_top_floor'] = (df['floor_numeric'] >= df['Szintek száma'].fillna(5) - 1).astype(int)
            df['floor_ratio'] = df['floor_numeric'] / df['Szintek száma'].fillna(5)
        else:
            df['is_ground_floor'] = 0
            df['is_top_floor'] = 0
            df['floor_ratio'] = np.nan
        
        # 5. Kerület prémium kategóriák
        premium_districts = [1, 2, 5, 6, 12]  # Belváros + prémium kerületek
        suburban_districts = [16, 17, 18, 19, 20, 21, 22, 23]  # Külváros
        
        if 'kerület' in df.columns:
            df['is_premium_district'] = df['kerület'].isin(premium_districts).astype(int)
            df['is_suburban'] = df['kerület'].isin(suburban_districts).astype(int)
        else:
            df['is_premium_district'] = 0
            df['is_suburban'] = 0
        
        # ===== INTERAKCIÓS FEATURE-ÖK =====
        
        # 6. Ár vs kerületi átlag arány (relatív drágaság)
        if 'price_per_m2' in df.columns and 'district_avg_price' in df.columns:
            df['price_vs_district_ratio'] = df['price_per_m2'] / df['district_avg_price']
        else:
            df['price_vs_district_ratio'] = np.nan
        
        # 7. Méret kategóriák
        if 'area_m2' in df.columns:
            df['size_category'] = pd.cut(df['area_m2'].fillna(50), 
                                        bins=[0, 40, 60, 80, 120, 1000],
                                        labels=['Kicsi', 'Közepes', 'Nagy', 'XL', 'Villa'])
        else:
            df['size_category'] = 'Ismeretlen'
        
        # 8. Energetikai hatékonyság számérték
        if 'Energetikai besorolás' in df.columns:
            energy_map = {'A+': 7, 'A': 6, 'B': 5, 'C': 4, 'D': 3, 'E': 2, 'F': 1, 'G': 0}
            df['energy_numeric'] = df['Energetikai besorolás'].map(energy_map).fillna(2)
        else:
            df['energy_numeric'] = 2
        
        # 9. Állapot számérték
        if 'Állapot' in df.columns:
            condition_map = {'Új építésű': 6, 'Újszerű': 5, 'Felújított': 4, 
                            'Jó állapotú': 3, 'Átlagos': 2, 'Felújítandó': 1}
            df['condition_numeric'] = df['Állapot'].map(condition_map).fillna(2)
        else:
            df['condition_numeric'] = 2
        
        # 10. Fűtés típus prémium
        if 'Fűtés' in df.columns:
            premium_heating = ['Mennyezeti hűtés-fűtés', 'Hőszivattyú', 'Gázkazán']
            df['premium_heating'] = df['Fűtés'].isin(premium_heating).astype(int)
        else:
            df['premium_heating'] = 0
        
        # ===== ÖSSZETETT FEATURE-ÖK =====
        
        # 11. Minőségi index (állapot + energetika + fűtés)
        df['quality_index'] = (df['condition_numeric'] + df['energy_numeric'] + 
                            df['premium_heating'] * 2) / 4
        
        # 12. Lokációs pontszám (kerület + emelet + outdoor space)
        df['location_score'] = (df['is_premium_district'] * 3 + 
                            (1 - df['is_suburban']) * 2 + 
                            df['floor_ratio'].fillna(0.5) + 
                            (df['outdoor_space'] > 0).astype(int))
        
        # 13. Modern features kombinációja
        df['modern_features'] = ((df['building_age'] < 20).astype(int) + 
                                df['premium_heating'] + 
                                (df['energy_numeric'] >= 5).astype(int))
        
        # 14. Praktikussági index
        df['practicality_index'] = (df['m2_per_room'].fillna(25) / 30 +  # Ideális ~30m2/szoba
                                    (df['area_m2'].fillna(0) >= 50).astype(int) +  # Minimum méret
                                    (df['has_balcony'] | df['has_terrace']).astype(int))
        
        return df

    def clean_and_prepare_data(self, df):
        """Adattisztítás és előkészítés - KIADÓ SPECIFIKUS (könnyített)"""
        # Kezdeti rekordok száma
        initial_count = len(df)
        
        # Alap feltételek
        conditions = [
            ('price', '>', 50),          # Minimum 50e Ft/hó
            ('price', '<', 2000),         # Maximum 2M Ft/hó
            ('area_m2', '>', 15),         # Minimum 15 m²
            ('area_m2', '<', 300),        # Maximum 300 m²
            ('price_per_m2', '>', 0.5),   # Minimum 500 Ft/m²/hó (új feltétel)
            ('price_per_m2', '<', 50),    # Maximum 50e Ft/m²/hó (új feltétel)
            ('location', 'contains', 'Budapest')
        ]
        
        # Dinamikus szűrés
        for col, op, value in conditions:
            if col in df.columns:
                if op == '>':
                    df = df[df[col] > value]
                elif op == '<':
                    df = df[df[col] < value]
                elif op == 'contains':
                    df = df[df[col].str.contains(value, na=False)]
        
        # Szűrt rekordok száma
        filtered_count = len(df)
        print(f"📊 Szűrés: {initial_count} → {filtered_count} rekord ({initial_count-filtered_count} eltávolítva)")
        
        return df

    def get_feature_columns(self):
        """Feature oszlopok definíciója - KIADÓ SPECIFIKUS"""
        # Numerikus feature-ök
        numerical_features = [
            'area_m2', 'floor_numeric', 'rooms', 'building_age', 'energy_numeric',
            'condition_numeric', 'quality_index', 'location_score', 'modern_features',
            'practicality_index', 'district_rank', 'm2_per_room', 'outdoor_space',
            'floor_ratio', 'district_avg_rent', 'Építés éve', 'Szintek száma',
            # Új kiadó specifikus numerikus feature-ök
            'rent_per_m2', 'deposit_ratio', 'min_rental_months', 'furniture_score'
        ]
        
        # Kategorikus feature-ök
        categorical_features = [
            'kerület', 'Típus', 'Állapot', 'Fűtés', 'Energetikai besorolás',
            'era', 'size_category', 'is_premium_district', 'is_suburban',
            'has_balcony', 'has_terrace', 'is_ground_floor', 'is_top_floor',
            'premium_heating', 
            # Új kiadó specifikus kategorikus feature-ök
            'has_deposit', 'utilities_included', 'is_new_building', 'available_immediately'
        ]
        
        return numerical_features, categorical_features
    
    def create_preprocessor(self, numerical_features, categorical_features):
        """Preprocessing pipeline létrehozása"""
        numerical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())  # Scaling hozzáadása
        ])
        
        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        
        preprocessor = ColumnTransformer([
            ('num', numerical_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)
        ])
        
        return preprocessor

    def preprocess_full_pipeline(self, df):
        """Teljes preprocessing pipeline - KIADÓ SPECIFIKUS"""
        print("🔧 Feature engineering...")
        
        # District prices előkészítés - FLEXIBILIS MEGKÖZELÍTÉS
        # Ellenőrizzük, hogy van-e kerületi átlagár adatunk
        if 'Lakások összesen átlagár, ezer Ft/m²' in self.district_prices.columns:
            self.district_prices['kerület'] = self.district_prices['Az ingatlan helye'].str.extract('(\\d+)').astype(float)
            district_avg = self.district_prices[['kerület', 'Lakások összesen átlagár, ezer Ft/m²']].copy()
            district_avg.iloc[:, 1] = district_avg.iloc[:, 1] / 1000  # Átváltás ezer Ft/m²-ről Ft/m²-re
            district_avg = district_avg.rename(columns={'Lakások összesen átlagár, ezer Ft/m²': 'district_avg_price'})
        else:
            # Ha nincs kerületi átlagár adat, hozzunk létre egy alapértelmezettet
            print("⚠️  Nincs kerületi átlagár adat, alapértelmezett értékek használata")
            district_avg = pd.DataFrame({
                'kerület': range(1, 24),
                'district_avg_price': [1.0] * 23  # Alapértelmezett érték
            })
        
        # Kerület kinyerése
        def extract_district(location):
            if pd.isna(location) or 'kerület' not in str(location):
                return None
            try:
                roman_nums = {'I':1, 'II':2, 'III':3, 'IV':4, 'V':5, 'VI':6, 'VII':7, 'VIII':8, 'IX':9, 'X':10,
                            'XI':11, 'XII':12, 'XIII':13, 'XIV':14, 'XV':15, 'XVI':16, 'XVII':17, 'XVIII':18,
                            'XIX':19, 'XX':20, 'XXI':21, 'XXII':22, 'XXIII':23}
                
                location_clean = str(location).replace('.', '').strip()
                for roman, num in roman_nums.items():
                    if roman in location_clean:
                        return num
                return None
            except:
                return None
        
        df['kerület'] = df['location'].apply(extract_district)
        
        # District prices merge
        df = pd.merge(df, district_avg, on='kerület', how='left')
        
        # Price per m2 számítás (ha van ár és terület adat)
        if 'price' in df.columns and 'area_m2' in df.columns:
            df['price_per_m2'] = df['price'] / df['area_m2']
        else:
            df['price_per_m2'] = np.nan
        
        # Emelet numerikus konverzió
        def parse_floor(floor_str):
            if pd.isna(floor_str):
                return 0
            floor_str = str(floor_str).lower()
            if 'földszint' in floor_str:
                return 0
            if 'félemelet' in floor_str:
                return 0.5
            if 'szint' in floor_str:
                return 0  # Alapértelmezett
            match = re.search(r'(\\d+)', floor_str)
            return int(match.group(1)) if match else 0
        
        df['floor_numeric'] = df['floor'].apply(parse_floor)
        
        # Adattisztítás - KIADÓ SPECIFIKUS (könnyített feltételek)
        df = self.clean_and_prepare_data(df)
        
        # Fejlett feature-ök létrehozása - KIADÓ SPECIFIKUS
        df = self.create_advanced_features(df)
        
        # Feature oszlopok meghatározása
        numerical_features, categorical_features = self.get_feature_columns()
        
        # Csak létező oszlopok megtartása
        existing_num = [f for f in numerical_features if f in df.columns]
        existing_cat = [f for f in categorical_features if f in df.columns]
        
        print(f"📊 Numerikus feature-ök: {len(existing_num)}")
        print(f"📊 Kategorikus feature-ök: {len(existing_cat)}")
        
        # Preprocessor létrehozása
        self.preprocessor = self.create_preprocessor(existing_num, existing_cat)
        
        # Target és feature-ök szétválasztása
        X = df[existing_num + existing_cat]
    
        # price_per_m2 legyen a célváltozó
        if 'price_per_m2' in df.columns:
            y = df['price_per_m2'] * 1000
            print("🎯 Célváltozó: price_per_m2")
        else:
            # Ha nincs price_per_m2, számoljuk ki
            df['price_per_m2'] = df['price'] / df['area_m2']
            y = df['price_per_m2'] * 1000
            print("🎯 Célváltozó: price_per_m2 (kiszámolva)")
        
        return df, X, y, existing_num, existing_cat
    
class AdvancedModelOptimizer:
    def __init__(self):
        self.models = {}
        self.best_model = None
        self.best_score = float('inf')
        
    def get_optimized_models(self):
        """Optimalizált modell konfigurációk"""
        return {
            "Random Forest Pro": RandomForestRegressor(),
            
            "XGBoost": xgb.XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1
            ),
            
            "LightGBM": lgb.LGBMRegressor(
                n_estimators=200,
                max_depth=8,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            ),
            
            "Extra Trees": ExtraTreesRegressor(
                n_estimators=200,
                max_depth=12,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            ),
            
            "Gradient Boosting Pro": GradientBoostingRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                random_state=42
            ),
            
            "Ridge": Ridge(alpha=1.0),
            "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
        }
    
    def hyperparameter_tuning(self, model, param_grid, X_train, y_train):
        """Hiperparaméter tuning"""
        grid_search = GridSearchCV(
            model, param_grid, cv=5, 
            scoring='neg_mean_squared_error', 
            n_jobs=-1, verbose=1
        )
        grid_search.fit(X_train, y_train)
        return grid_search.best_estimator_, grid_search.best_score_
    
    def evaluate_models(self, X_train, X_test, y_train, y_test):
        """Modellek kiértékelése - price_per_m2 SPECIFIKUS"""
        models = self.get_optimized_models()
        results = {}
        
        print("🚀 Modellek tesztelése...")
        print(f"🎯 Célváltozó: price_per_m2 (Ft/m²/hó)")
        
        # Átlagos price_per_m2 számolása értelmezéshez
        avg_price_per_m2 = y_train.mean()
        print(f"📊 Átlagos price_per_m2: {avg_price_per_m2:.2f} Ft/m²/hó")
        
        for name, model in models.items():
            print(f"\n📈 {name} tesztelése...")
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, 
                                        scoring='neg_mean_squared_error', n_jobs=-1)
            cv_rmse = np.sqrt(-cv_scores)
            
            # Modell tanítása
            model.fit(X_train, y_train)
            
            # Predikciók
            y_pred = model.predict(X_test)
            
            # Metrikák
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            
            # Relatív hiba számolása
            relative_error = (rmse / avg_price_per_m2) * 100
            
            results[name] = {
                'CV_RMSE_mean': cv_rmse.mean(),
                'CV_RMSE_std': cv_rmse.std(),
                'Test_RMSE': rmse,
                'Test_MAE': mae,
                'R2_Score': r2,
                'Relative_Error_%': relative_error,
                'Model': model,
                'Predictions': y_pred
            }
            
            if rmse < self.best_score:
                self.best_score = rmse
                self.best_model = (name, model)
            
            print(f"  R² score: {r2:.4f}")
            print(f"  CV RMSE: {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}")
            print(f"  Test RMSE: {rmse:.3f} Ft/m²/hó")
            print(f"  Relatív hiba: {relative_error:.1f}%")
            print(f"  Test MAE: {mae:.3f} Ft/m²/hó")
        
        return results
    
    def plot_results(self, results, feature_names):
        """Eredmények vizualizációja"""
        # Model összehasonlítás
        results_df = pd.DataFrame({k: v for k, v in results.items() if k not in ['Model', 'Predictions', 'y_test']}).T
        results_df = results_df.sort_values('Test_RMSE')
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # RMSE összehasonlítás
        sns.barplot(data=results_df.reset_index(), x='index', y='Test_RMSE', ax=axes[0,0])
        axes[0,0].set_title('Model RMSE Comparison')
        axes[0,0].tick_params(axis='x', rotation=45)
        
        # R² összehasonlítás
        sns.barplot(data=results_df.reset_index(), x='index', y='R2_Score', ax=axes[0,1])
        axes[0,1].set_title('Model R² Comparison')
        axes[0,1].tick_params(axis='x', rotation=45)
        
        # Feature importance (legjobb modell)
        best_model_name = results_df.index[0]
        best_model = results[best_model_name]['Model']
        
        if hasattr(best_model, 'feature_importances_'):
            importance = pd.Series(best_model.feature_importances_, index=feature_names)
            top_features = importance.sort_values(ascending=False).head(15)
            
            sns.barplot(x=top_features.values, y=top_features.index, ax=axes[1,0])
            axes[1,0].set_title(f'Top 15 Features - {best_model_name}')
            axes[1,0].set_xlabel('Importance')
        
        # Actual vs Predicted
        y_test_actual = results['y_test']
        y_pred = results[best_model_name]['Predictions']
        sns.scatterplot(x=y_test_actual, y=y_pred, ax=axes[1,1], alpha=0.6)
        axes[1,1].plot([y_test_actual.min(), y_test_actual.max()], 
                      [y_test_actual.min(), y_test_actual.max()], 'k--')
        axes[1,1].set_xlabel('Actual Price/m²')
        axes[1,1].set_ylabel('Predicted Price/m²')
        axes[1,1].set_title(f'Actual vs Predicted - {best_model_name}')
        
        plt.tight_layout()
        plt.show()
        
        return results_df
    
class FeatureSelector:
    def __init__(self, preprocessor, optimizer, X, y_train, y_test):
        self.preprocessor = preprocessor
        self.optimizer = optimizer
        self.X = X  # Ez most már helyesen van tárolva
        self.y_train = y_train
        self.y_test = y_test

    def evaluate_with_feature_subset(self, feature_subset):
        """Kiértékeli a legjobb modellt egy adott feature alhalmazon."""
        # Itt már a self.X-et használjuk az átadott X helyett
        X_subset = self.X[feature_subset].copy()
        
        # A pipeline illesztése és transzformálása
        # Fontos: A preprocessor-t itt kell létrehozni, hogy minden iterációban újragenerálódjon
        # a megfelelő feature készletre
        numerical_features = [f for f in feature_subset if f in X_subset.select_dtypes(include=[np.number]).columns]
        categorical_features = [f for f in feature_subset if f in X_subset.select_dtypes(include=['object']).columns]
        
        # Új preprocessor létrehozása a feature subset-hez
        subset_preprocessor = ColumnTransformer([
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), numerical_features),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), categorical_features)
        ])
        
        # Preprocessing
        X_train_processed = subset_preprocessor.fit_transform(X_subset.loc[self.y_train.index])
        X_test_processed = subset_preprocessor.transform(X_subset.loc[self.y_test.index])

        # A legjobb modell kiválasztása és kiértékelése
        models = self.optimizer.get_optimized_models()
        best_name, best_model = "LightGBM", models["LightGBM"]
        
        best_model.fit(X_train_processed, self.y_train)
        y_pred = best_model.predict(X_test_processed)
        rmse = np.sqrt(mean_squared_error(self.y_test, y_pred))
        
        return rmse

    def iterative_backward_selection(self, all_features, advanced_features):
        """Iteratív hátrafelé eliminációs feature selection."""
        best_rmse = float('inf')
        best_feature_set = all_features.copy()
        
        print("\n🔍 Iteratív feature selection kezdődik...")
        
        # Először kiértékeljük a teljes feature készletet
        print("🧪 Teljes feature készlet kiértékelése...")
        full_rmse = self.evaluate_with_feature_subset(all_features)
        print(f"   Teljes feature készlet RMSE: {full_rmse:.3f}")
        best_rmse = full_rmse
        
        # Iterálás minden advanced feature-ön
        for feature_to_remove in advanced_features:
            if feature_to_remove not in all_features:
                continue
                
            print(f"➖ Eltávolítva: '{feature_to_remove}'")
            
            # Készítsd el a feature halmazt a kihagyott feature nélkül
            current_features = [f for f in all_features if f != feature_to_remove]
            
            # Keresztvalidációs RMSE számítása a teljesítmény mérésére
            try:
                current_rmse = self.evaluate_with_feature_subset(current_features)
            except Exception as e:
                print(f"   ❌ Hiba: {e}")
                continue

            print(f"   Új RMSE: {current_rmse:.3f}")

            if current_rmse < best_rmse:
                improvement = (best_rmse - current_rmse) / best_rmse * 100
                print(f"   ✅ Javulás: {improvement:.2f}%! Új legjobb feature készlet.")
                best_rmse = current_rmse
                best_feature_set = current_features
            else:
                degradation = (current_rmse - best_rmse) / best_rmse * 100
                print(f"   ❌ Romlás: {degradation:.2f}%")
                
        return best_feature_set, best_rmse
def interpret_rmse_price_per_m2(rmse_value, avg_price_per_m2):
    """
    Segédfüggvény az RMSE értelmezéséhez price_per_m2 esetén
    """
    relative_error = (rmse_value / avg_price_per_m2) * 100
    
    print(f"\n📊 RMSE ÉRTELMEZÉSE:")
    print(f"• Abszolút RMSE: {rmse_value:.2f} Ft/m²/hó")
    print(f"• Átlagos ár: {avg_price_per_m2:.2f} Ft/m²/hó")
    print(f"• Relatív hiba: {relative_error:.1f}%")
    
    if relative_error < 10:
        print("✅ Kiváló modell - praktikusan használható")
    elif relative_error < 20:
        print("✅ Jó modell - elfogadható pontosság")
    elif relative_error < 30:
        print("⚠️  Átlagos modell - iránytrendek mutatása")
    else:
        print("❌ Gyenge modell - túl sok a zaj")
    
    print(f"\n🏢 Gyakorlati példa:")
    print(f"Egy 60 m²-es lakásnál ez átlagos {rmse_value * 60:.0f} Ft/hó becslési hibát jelent")

def main_advanced_analysis():
    """Fő elemzési függvény - KIADÓ SPECIFIKUS"""
    print("📊 Fejlett BÉRLETI ármodell építés kezdődik...")
    print("🎯 Célváltozó: price_per_m2 (Ft/m²/hó)")
    
    # Adatok betöltése - MÓDOSÍTOTT FÁJLNÉV
    df = pd.read_csv("zenga_rentals_details_optimized.csv")  # MÓDOSÍTVA
    district_prices_path = "Ingatlanadattár, 2023 - Budapest.csv"
    
    print(f"📋 Betöltött rekordok száma: {len(df)}")
    
    # Preprocessing - KIADÓ SPECIFIKUS
    preprocessor = AdvancedRealEstatePreprocessor(district_prices_path)
    df_processed, X, y, num_features, cat_features = preprocessor.preprocess_full_pipeline(df)
    
    print(f"✅ Tisztított rekordok száma: {len(df_processed)}")
    print(f"🎯 Feature-ök száma: {len(num_features) + len(cat_features)}")
    print(f"💰 Átlagos bérleti díj: {df_processed['price'].mean():.0f} Ft/hó")
    print(f"📏 Átlagos terület: {df_processed['area_m2'].mean():.1f} m²")
    print(f"🏷️  Átlagos price_per_m2: {y.mean():.2f} Ft/m²/hó")
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Preprocessing alkalmazása
    X_train_processed = preprocessor.preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.preprocessor.transform(X_test)
    
    # Feature nevek létrehozása
    try:
        cat_encoder = preprocessor.preprocessor.named_transformers_['cat'].named_steps['onehot']
        cat_feature_names = cat_encoder.get_feature_names_out(cat_features)
        feature_names = num_features + list(cat_feature_names)
    except:
        feature_names = [f'feature_{i}' for i in range(X_train_processed.shape[1])]
    
    # Modellek optimalizálása a teljes feature készlettel (alapvonal)
    optimizer = AdvancedModelOptimizer()
    results = optimizer.evaluate_models(X_train_processed, X_test_processed, y_train, y_test)
    
    # Eredmények tárolása y_test-tel
    results['y_test'] = y_test
    
    # Vizualizáció (még a teljes feature készlettel)
    results_df = optimizer.plot_results(results, feature_names)
    
    # Legjobb modell kiválasztása
    best_name, best_model = optimizer.best_model
    print(f"\n🏆 Legjobb modell (alapvonal): {best_name}")
    print(f"📈 RMSE (alapvonal): {optimizer.best_score:.3f}")

    # A modell kiértékelése után
    interpret_rmse_price_per_m2(optimizer.best_score, y.mean())

    # ===== FEATURE SELECTION =====
    # A fejlett feature-ök azonosítása
    advanced_features = [
        'quality_index', 'location_score', 'modern_features', 'practicality_index',
        'district_rank', 'price_vs_district_ratio', 'm2_per_room', 'outdoor_space',
        'is_premium_district', 'is_suburban', 'has_balcony', 'has_terrace',
        'is_ground_floor', 'is_top_floor', 'floor_ratio', 'building_age',
        'energy_numeric', 'condition_numeric', 'premium_heating', 'era',
        'size_category', 'size_zscore', 'is_size_outlier'
    ]
    
    # Csak azokat tartsd meg, amelyek valóban léteznek az adatkészletedben
    advanced_features = [f for f in advanced_features if f in X.columns]
    
    # Feature selector inicializálása - most már helyesen átadjuk X-et
    feature_selector = FeatureSelector(preprocessor, optimizer, X, y_train, y_test)
    
    # Összes feature
    all_features = num_features + cat_features
    
    final_features, final_rmse = feature_selector.iterative_backward_selection(
        all_features, advanced_features
    )
    
    print("\n✅ Feature Selection befejezve.")
    print(f"🏆 Végső legjobb RMSE: {final_rmse:.3f}")
    print(f"🎯 Optimális feature-ök száma: {len(final_features)}")
    print(f"Listájuk: {final_features}")
    
    # Feature importance részletes elemzés
    if hasattr(best_model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': best_model.feature_importances_
        }).sort_values('Importance', ascending=False)
        
        print("\n🔝 Top 10 legfontosabb feature:")
        for i, row in importance_df.head(10).iterrows():
            print(f"  {row['Feature']}: {row['Importance']:.4f}")

    import joblib

    # Legjobb modell mentése
    best_model_name, best_model = optimizer.best_model
    joblib.dump(best_model, 'best_rental_price_model.pkl')
    joblib.dump(preprocessor.preprocessor, 'preprocessor.pkl')
    joblib.dump((num_features, cat_features), 'feature_columns.pkl')

    print(f"💾 Modell elmentve: {best_model_name}")
    
    return df_processed, preprocessor, results, feature_names

In [ ]:
# Futtatás
if __name__ == "__main__":
    df_final, preprocessor_final, results_final, features_final = main_advanced_analysis()

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# Modell betöltése
try:
    model = joblib.load('best_rental_price_model.pkl')
    preprocessor = joblib.load('preprocessor.pkl')
    existing_num, existing_cat = joblib.load('feature_columns.pkl')
    print("✅ Modell betöltve sikeresen!")
except:
    print("❌ Modell betöltése sikertelen. Futtasd le először a fő elemzést!")
    model = None
    preprocessor = None

# Interaktív elemek létrehozása
def create_rental_prediction_widget():
    # Számszerű bemenetek
    area_widget = widgets.FloatSlider(
        value=60, min=15, max=200, step=5, description='Terület (m²):',
        style={'description_width': 'initial'}
    )
    
    rooms_widget = widgets.IntSlider(
        value=2, min=1, max=6, description='Szobák száma:',
        style={'description_width': 'initial'}
    )
    
    floor_widget = widgets.IntSlider(
        value=2, min=0, max=10, description='Emelet:',
        style={'description_width': 'initial'}
    )
    
    building_age_widget = widgets.IntSlider(
        value=20, min=0, max=100, description='Épület kora (év):',
        style={'description_width': 'initial'}
    )
    
    # Legördülő menük
    district_widget = widgets.Dropdown(
        options=[(f'{i}. kerület', i) for i in range(1, 24)],
        value=6, description='Kerület:',
        style={'description_width': 'initial'}
    )
    
    condition_widget = widgets.Dropdown(
        options=['Új építésű', 'Újszerű', 'Felújított', 'Jó állapotú', 'Átlagos', 'Felújítandó'],
        value='Jó állapotú', description='Állapot:',
        style={'description_width': 'initial'}
    )
    
    heating_widget = widgets.Dropdown(
        options=['Gázkazán', 'Távfűtés', 'Hőszivattyú', 'Elektromos', 'Cirkó', 'Egyéb'],
        value='Gázkazán', description='Fűtés:',
        style={'description_width': 'initial'}
    )
    
    energy_widget = widgets.Dropdown(
        options=['A+', 'A', 'B', 'C', 'D', 'E', 'F', 'G'],
        value='C', description='Energetikai:',
        style={'description_width': 'initial'}
    )
    
    balcony_widget = widgets.Checkbox(
        value=True, description='Van erkély',
        style={'description_width': 'initial'}
    )
    
    terrace_widget = widgets.Checkbox(
        value=False, description='Van terasz',
        style={'description_width': 'initial'}
    )
    
    # Gomb az előrejelzéshez
    predict_button = widgets.Button(
        description="💰 Árbecslés készítése",
        button_style='success',
        tooltip='Kattints az árbecsléshez'
    )
    
    # Eredmény megjelenítése
    result_output = widgets.Output()
    
    def predict_price(b):
        if model is None or preprocessor is None:
            with result_output:
                print("❌ Először töltsd be a modellt!")
            return
        
        # Adatok összeállítása
        input_data = {
            'area_m2': area_widget.value,
            'rooms': rooms_widget.value,
            'floor_numeric': floor_widget.value,
            'building_age': building_age_widget.value,
            'kerület': district_widget.value,
            'Állapot': condition_widget.value,
            'Fűtés': heating_widget.value,
            'Energetikai besorolás': energy_widget.value,
            'has_balcony': 1 if balcony_widget.value else 0,
            'has_terrace': 1 if terrace_widget.value else 0,
        }
        
        # Hiányzó értékek kitöltése
        for feature in existing_num + existing_cat:
            if feature not in input_data:
                if feature in existing_num:
                    input_data[feature] = 0  # Numerikus alapérték
                else:
                    input_data[feature] = 'Ismeretlen'  # Kategorikus alapérték
        
        # DataFrame készítése
        input_df = pd.DataFrame([input_data])
        
        try:
            # Preprocesszálás
            X_processed = preprocessor.transform(input_df)
            
            # Előrejelzés
            price_per_m2_pred = model.predict(X_processed)[0]
            total_price_pred = price_per_m2_pred * area_widget.value
            
            with result_output:
                result_output.clear_output()
                print("🏠 **Albérlet árbecslés**")
                print("=" * 40)
                print(f"📐 Terület: {area_widget.value} m²")
                print(f"🚪 Szobák: {rooms_widget.value}")
                print(f"🏢 Kerület: {district_widget.value}.")
                print(f"🔧 Állapot: {condition_widget.value}")
                print("=" * 40)
                print(f"💰 **Becsült ár/m²/hó: {price_per_m2_pred:.0f} Ft**")
                print(f"💵 **Becsült havi bérleti díj: {total_price_pred:.0f} Ft**")
                print("=" * 40)
                
                # Árösszehasonlítás
                avg_price_per_m2 = 2500  # Példa átlagár, helyettesítsd valós adattal
                price_ratio = (price_per_m2_pred / avg_price_per_m2 - 1) * 100
                
                if price_ratio > 20:
                    print("📈 Ez az ár jelentősen magasabb az átlagnál")
                elif price_ratio > 10:
                    print("📈 Ez az ár magasabb az átlagnál")
                elif price_ratio < -20:
                    print("📉 Ez az ár jelentősen alacsonyabb az átlagnál")
                elif price_ratio < -10:
                    print("📉 Ez az ár alacsonyabb az átlagnál")
                else:
                    print("📊 Ez az ár közel van az átlaghoz")
                    
        except Exception as e:
            with result_output:
                print(f"❌ Hiba történt: {str(e)}")
    
    predict_button.on_click(predict_price)
    
    # UI elrendezés
    left_panel = widgets.VBox([
        area_widget,
        rooms_widget,
        floor_widget,
        building_age_widget,
        district_widget
    ])
    
    right_panel = widgets.VBox([
        condition_widget,
        heating_widget,
        energy_widget,
        balcony_widget,
        terrace_widget
    ])
    
    input_panel = widgets.HBox([left_panel, right_panel])
    
    return widgets.VBox([
        widgets.HTML("<h2>🏠 Albérlet Árbecslő</h2>"),
        input_panel,
        predict_button,
        result_output
    ])

# Widget megjelenítése
if model is not None:
    rental_widget = create_rental_prediction_widget()
    display(rental_widget)
else:
    print("❌ Először futtasd le a modell tanítását!")